In [ ]:
import copy
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

In [ ]:
data = pd.read_csv(r"C:\Users\ykila\Desktop\iCloud\iCloudDrive\Projects\Active\Code\Python\LLM\Aalto\Dataframe.csv")
data["log_pSat"] = np.log10(data["pSat_Pa"])  # compress 10+ orders of magnitude into a tractable range

In [ ]:
# Generate ECFP4 (Morgan radius=2) fingerprints from SMILES — substructural signal not present in RDKit descriptors
def smiles_to_ecfp(smi, n_bits=2048, radius=2):
    arr = np.zeros(n_bits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps = np.stack([smiles_to_ecfp(s) for s in data["SMILES"]])
fp_df = pd.DataFrame(fps, columns=[f"fp_{i}" for i in range(fps.shape[1])], index=data.index)
print(f"generated {fps.shape[1]}-bit ECFP4 for {fps.shape[0]} molecules")

In [ ]:
X = fps  # ECFP4 fingerprints are the only feature set across all four versions
y = data["log_pSat"].values
print(f"X shape: {X.shape}  (pure {X.shape[1]}-bit ECFP4)")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )


In [ ]:
# Carve 10% off training to use as validation. Early stopping, best-state
# selection and best-vs-SWA pick all run on val so X_test stays unobserved
# until the final ensemble line.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).unsqueeze(1),
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).unsqueeze(1),
)

In [ ]:
train_dl = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"


def make_model():
    """Same architecture as v2. Wrapped in a factory so the seed-ensemble loop
    can build a fresh, independently-initialised model each pass."""
    return nn.Sequential(
        nn.Linear(X_tr.shape[1], 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1),
    ).to(device)

In [ ]:
loss_fn = nn.L1Loss()  # match the MAE metric directly

In [ ]:
@torch.no_grad()
def predict(m, dl):
    m.eval()
    out = []
    for x, _ in dl:
        out.append(m(x.to(device)).cpu().numpy())
    return np.vstack(out).flatten()


def train_one(seed, max_epochs=120, patience=15, swa_start=30):
    """Train one MLP. Same training discipline as v2 (val-driven early stopping,
    AdamW, plateau LR), plus SWA. Returns whichever of (best-epoch weights) or
    (SWA-averaged weights) gives a lower validation MAE — selection happens on
    val, not test."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    swa_model = torch.optim.swa_utils.AveragedModel(model)

    best_val_mae = float("inf")
    best_state = None
    since_best = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch >= swa_start:
            swa_model.update_parameters(model)

        # Validation pass — drives scheduler + early stopping
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                preds.append(model(x))
                trues.append(y)
        val_mae = mean_absolute_error(torch.cat(trues).cpu().numpy(),
                                      torch.cat(preds).cpu().numpy())
        scheduler.step(val_mae)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = copy.deepcopy(model.state_dict())
            since_best = 0
        else:
            since_best += 1
            if since_best >= patience:
                break

    # Pick best-epoch vs SWA on validation only
    model.load_state_dict(best_state)
    best_val = mean_absolute_error(y_val, predict(model, val_dl))
    swa_val = mean_absolute_error(y_val, predict(swa_model, val_dl))

    if swa_val < best_val:
        chosen, val_mae, kind = swa_model, swa_val, "SWA"
    else:
        chosen, val_mae, kind = model, best_val, "best"

    # Final test prediction — single use of test set per seed, no decisions made on it
    test_preds = predict(chosen, test_dl)
    return test_preds, val_mae, kind


SEEDS = [0, 1, 2, 3, 4]
seed_test_preds = []
for s in SEEDS:
    p, val_mae, kind = train_one(s)
    print(f"seed {s}: val MAE = {val_mae:.4f}  ({kind})")
    seed_test_preds.append(p)

nn_test_preds = np.mean(seed_test_preds, axis=0)
print(f"\n{len(SEEDS)}-seed NN ensemble test MAE: {mean_absolute_error(y_test, nn_test_preds):.4f}")